# Семинар 7. Подзапросы и общие табличные выражения

На этом занятии вы будете вкладывать один запрос в другой: сравнивать строки со значением,
посчитанным по всей таблице, проверять, есть ли у строки пара в другой таблице,
и разбивать сложный запрос на именованные шаги с помощью `WITH`.

**Как устроен ноутбук.** Каждая задача — описание и ячейка, которая начинается с `%%sql`.
Всё, что написано в такой ячейке ниже первой строки, отправляется в PostgreSQL,
а результат показывается таблицей.

**Оценка.** Всего 7 задач на **10 баллов**, баллы указаны в заголовке каждой задачи.

**База.** PostgreSQL 17 в Docker, демобаза авиаперевозок Postgres Pro (версия 2016 года).
Как поднять базу, открыть ноутбук и подключиться через DBeaver, написано в файле `README.md`
в корне репозитория.

Все таблицы лежат в схеме `bookings`, в запросах пишем имя вместе со схемой: `bookings.flights`.

### Таблицы демобазы, которые понадобятся

| Таблица | Одна строка — это | Столбцы, которые нужны в задачах |
|---|---|---|
| `bookings.aircrafts` | модель самолёта | `aircraft_code` — код модели, `model` — название, `range` — дальность полёта, км |
| `bookings.seats` | место в салоне модели самолёта | `aircraft_code` — код модели, `seat_no` — номер места |
| `bookings.flights` | рейс в конкретный день | `flight_id`, `departure_airport` и `arrival_airport` — коды аэропортов вылета и прилёта, `aircraft_code` — код модели, `status` — статус, `scheduled_departure` и `actual_departure` — вылет по расписанию и фактический |
| `bookings.ticket_flights` | перелёт по билету на конкретном рейсе | `flight_id` — рейс, `amount` — стоимость перелёта |
| `bookings.airports` | аэропорт | `airport_code` — код из трёх букв, `airport_name` — название, `city` — город |

Как таблицы связаны между собой:

- `flights.aircraft_code` и `seats.aircraft_code` → `aircrafts.aircraft_code`;
- `flights.departure_airport` и `flights.arrival_airport` → `airports.airport_code`;
- `ticket_flights.flight_id` → `flights.flight_id`.

Данные сформированы на момент 13.10.2016 17:00: у более поздних рейсов фактического вылета ещё нет (`NULL`).

**Схема демобазы.** Таблицы, их ключи и связи; подробное описание столбцов — в [`db/schema.md`](../db/schema.md).

![Схема демобазы авиаперевозок](../db/schema.svg)

In [ ]:
import sys; sys.path.insert(0, "../jupyter")   # магия %%sql лежит в папке jupyter репозитория
%load_ext sql_magic

### Проверка подключения
Если ячейка ниже показала пять моделей самолётов — всё готово к работе.

In [ ]:
%%sql
SELECT * FROM bookings.aircrafts LIMIT 5;

# Задачи

**Памятка**

| Что | Как пишется |
|---|---|
| Сравнить строку со значением по всей таблице | `WHERE x > (SELECT AVG(x) FROM …)` — скалярный подзапрос |
| То же значение вывести рядом со строкой | `(SELECT AVG(x) FROM …) AS avg_x` в `SELECT` |
| Сравнить группу со значением по всей таблице | `HAVING AVG(x) > (SELECT AVG(x) FROM …)` |
| Значение, своё для каждой строки | коррелированный подзапрос: `(SELECT … FROM t2 WHERE t2.key = t1.key)` |
| Есть ли пара в другой таблице | `WHERE EXISTS (SELECT 1 FROM … WHERE …)`, `WHERE x IN (SELECT …)` |
| Нет ли пары в другой таблице | `NOT EXISTS`, `NOT IN`, `LEFT JOIN … WHERE правая.ключ IS NULL` |
| Именованный шаг запроса | `WITH имя AS (SELECT …) SELECT … FROM имя;` |
| Задержка вылета в минутах | `ROUND(EXTRACT(EPOCH FROM actual_departure - scheduled_departure) / 60)` |

Несколько шагов `WITH` пишутся через запятую: `WITH a AS (…), b AS (…) SELECT …`.

### Задача 1. Модели с дальностью выше средней (1 балл)

Из таблицы моделей самолётов выведите модели, дальность полёта которых выше средней дальности
по всем моделям.

Выведите:

- **aircraft_code** — код модели;
- **model** — название модели;
- **range** — дальность полёта;
- **avg_range** — средняя дальность по всем моделям, округлённая до двух знаков;
- **diff** — разница между дальностью модели и средней дальностью, округлённая до двух знаков.

Отсортируйте по дальности от большей к меньшей.

In [ ]:
%%sql
-- Задача 1. Напишите запрос

### Задача 2. Доля рейсов в каждом статусе (1 балл)

По таблице рейсов для каждого статуса посчитайте число рейсов, общее число рейсов во всех статусах
и долю рейсов в этом статусе от всех рейсов.

Выведите:

- **status** — статус рейса;
- **cnt_flights** — число рейсов в статусе;
- **cnt_all** — общее число рейсов;
- **pct** — доля рейсов в статусе в процентах, округлённая до двух знаков.

Отсортируйте по числу рейсов от большего к меньшему.

In [ ]:
%%sql
-- Задача 2. Напишите запрос

### Задача 3. Модели с дорогими перелётами (1,5 балла)

По таблицам перелётов, рейсов и моделей самолётов найдите модели, у которых средняя стоимость
перелёта выше **30 000** рублей.

Выведите:

- **model** — название модели;
- **avg_amount** — средняя стоимость перелёта, округлённая до целого.

Отсортируйте по средней стоимости от большей к меньшей.

In [ ]:
%%sql
-- Задача 3. Напишите запрос

### Задача 4. Аэропорты с задержкой выше средней (1,5 балла)

Найдите аэропорты вылета, у которых средняя задержка вылета выше средней задержки по всем рейсам.
Задержку считайте в минутах по формуле из памятки, только по вылетевшим рейсам
(у которых заполнен `actual_departure`).

Выведите:

- **departure_airport** — код аэропорта вылета;
- **avg_delay** — средняя задержка вылета из аэропорта, округлённая до одного знака.

Отсортируйте по средней задержке от большей к меньшей.

In [ ]:
%%sql
-- Задача 4. Напишите запрос

### Задача 5. Модели, которые выполнили больше 5000 рейсов (1,5 балла)

Выведите модели самолётов, которые выполнили **больше 5000** рейсов — решите задачу двумя способами,
в двух ячейках:

1. с коррелированным подзапросом;
2. с CTE (`WITH`).

Выведите **aircraft_code** и **model**, отсортируйте по коду модели.

In [ ]:
%%sql
-- Задача 5.1. Напишите запрос

In [ ]:
%%sql
-- Задача 5.2. Напишите запрос

### Задача 6. Большие самолёты, которые часто летают (1,5 балла)

Найдите модели самолётов, у которых в салоне **больше 100** мест и которые выполнили **больше 1000** рейсов.
Места лежат в таблице `bookings.seats`, рейсы — в `bookings.flights`: посчитайте каждое число
отдельно, в своём CTE, а потом соедините результаты с таблицей моделей.

Выведите:

- **model** — название модели;
- **cnt_seats** — число мест;
- **cnt_flights** — число рейсов.

Отсортируйте по числу мест от большего к меньшему.

In [ ]:
%%sql
-- Задача 6. Напишите запрос

### Задача 7. Аэропорты, куда нет рейсов из Москвы (2 балла)

Найдите аэропорты, в которые нет ни одного рейса из Москвы. У Москвы три аэропорта:
`SVO`, `DME` и `VKO`.

Решите задачу тремя способами, в трёх ячейках:

1. с `NOT EXISTS`;
2. с `NOT IN`;
3. с `LEFT JOIN`.

Выведите **airport_code**, **airport_name**, **city**, отсортируйте по коду аэропорта.

In [ ]:
%%sql
-- Задача 7.1. Напишите запрос

In [ ]:
%%sql
-- Задача 7.2. Напишите запрос

In [ ]:
%%sql
-- Задача 7.3. Напишите запрос